# Examen Final - Caso de Estudio 3: Aprendizaje Semi-Supervisado y Q-Learning Logístico
Este notebook simula un caso práctico de examen final centrado en **Modelos Avanzados**:
1. **Aprendizaje Semi-Supervisado (Clustering + Clasificación):**
   - Simulación de un escenario real donde etiquetar datos es costoso.
   - Baseline supervisado con muy pocos datos etiquetados.
   - Uso de K-Means para encontrar imágenes representativas.
   - Propagación de etiquetas (**Label Propagation**) y re-entrenamiento del clasificador final.
2. **Aprendizaje por Refuerzo (Método Q-Learning):**
   - Implementación de un entorno de cuadrícula (Gridworld) para simular navegación logística.
   - Definición de la Q-Table y actualización mediante la **Ecuación de Bellman**.
   - Balance exploración vs. explotación con **Epsilon-Greedy**.
3. **Banco de Preguntas Teórico-Prácticas de Examen** al final del notebook.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')


## 1. Aprendizaje Semi-Supervisado (Label Propagation)


In [ ]:
# Cargar dataset de dígitos de Scikit-Learn
X_digits, y_digits = load_digits(return_X_y=True)

# Partición en entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(X_digits, y_digits, test_size=0.30, random_state=42)

print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)


### Baseline Supervisado (Solo 50 ejemplos etiquetados)


In [ ]:
# Asumimos que de los 1257 ejemplos de Train, solo tenemos etiquetas para los primeros 50
n_labeled = 50

# Modelo entrenado únicamente con los 50 datos etiquetados
lr_baseline = LogisticRegression(max_iter=5000, random_state=42)
lr_baseline.fit(X_train[:n_labeled], y_train[:n_labeled])

accuracy_base = accuracy_score(y_test, lr_baseline.predict(X_test))
print(f"Accuracy del modelo Baseline Supervisado (50 datos): {accuracy_base * 100:.2f}%")


### Enfoque Semi-Supervisado con Clustering (K-Means)


In [ ]:
# 1. Agrupamos los datos de entrenamiento en 50 clusters
k_clusters = 50
kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init='auto')
X_digits_dist = kmeans.fit_transform(X_train)

# 2. Identificamos el índice de la imagen más cercana al centroide de cada uno de los 50 clusters
representative_digit_idx = np.argmin(X_digits_dist, axis=0)
X_representative_digits = X_train[representative_digit_idx]

# 3. 'Etiquetamos a mano' estas 50 imágenes representativas (obtenemos sus etiquetas reales)
y_representative_labels = y_train[representative_digit_idx]

# 4. Entrenamos un modelo supervisado usando SOLO estos 50 representantes estratégicos
lr_representative = LogisticRegression(max_iter=5000, random_state=42)
lr_representative.fit(X_representative_digits, y_representative_labels)

accuracy_rep = accuracy_score(y_test, lr_representative.predict(X_test))
print(f"Accuracy entrenando con 50 REPRESENTANTES elegidos por K-Means: {accuracy_rep * 100:.2f}%")


### Propagación de Etiquetas (Label Propagation)


In [ ]:
# 5. Propagamos la etiqueta de cada representante a todas las demás imágenes que cayeron en su mismo cluster
y_train_propagated = np.empty(len(X_train), dtype=np.int32)
for i in range(k_clusters):
    # Asignamos la etiqueta del representante del cluster i a todos los miembros de ese cluster
    y_train_propagated[kmeans.labels_ == i] = y_representative_labels[i]

# 6. Entrenamos el modelo final con todo el set de entrenamiento ahora pseudo-etiquetado
lr_propagated = LogisticRegression(max_iter=5000, random_state=42)
lr_propagated.fit(X_train, y_train_propagated)

accuracy_final = accuracy_score(y_test, lr_propagated.predict(X_test))
print(f"Accuracy final tras Propagación de Etiquetas a todo el dataset: {accuracy_final * 100:.2f}%")


## 2. Aprendizaje por Refuerzo: Q-Learning


In [ ]:
# Definición de un entorno simple GridWorld (Rejilla de 4x4)
# S = Inicio (0,0), G = Meta (3,3), H = Trampa (1,2 y 2,1)
# El agente debe navegar desde S hasta G evitando las trampas.

class GridWorld:
    def __init__(self):
        self.grid_size = 4
        self.state = (0, 0)
        self.goal = (3, 3)
        self.traps = [(1, 2), (2, 1)]
        
    def reset(self):
        self.state = (0, 0)
        return self.state_to_index(self.state)
        
    def state_to_index(self, state):
        return state[0] * self.grid_size + state[1]
        
    def step(self, action):
        # Acciones: 0 = Arriba, 1 = Abajo, 2 = Izquierda, 3 = Derecha
        r, c = self.state
        if action == 0:   r = max(0, r - 1)
        elif action == 1: r = min(self.grid_size - 1, r + 1)
        elif action == 2: c = max(0, c - 1)
        elif action == 3: c = min(self.grid_size - 1, c + 1)
        
        self.state = (r, c)
        
        # Evaluar recompensa y fin de episodio
        if self.state == self.goal:
            return self.state_to_index(self.state), 10, True # Meta alcanzada
        elif self.state in self.traps:
            return self.state_to_index(self.state), -10, True # Caída en trampa
        else:
            return self.state_to_index(self.state), -1, False # Penalización por paso


In [ ]:
# Parámetros del algoritmo Q-Learning
env = GridWorld()
num_states = 16 # 4x4
num_actions = 4 # Arriba, Abajo, Izquierda, Derecha

# Inicializamos la Q-Table en 0
Q_table = np.zeros((num_states, num_actions))

alpha = 0.1     # Tasa de aprendizaje
gamma = 0.9     # Factor de descuento
epsilon = 0.8   # Probabilidad de exploración inicial
decay_rate = 0.99
num_episodios = 500

recompensas_por_episodio = []

for ep in range(num_episodios):
    state_idx = env.reset()
    done = False
    total_reward = 0
    
    while not done:
        # Acción Epsilon-Greedy
        if np.random.uniform(0, 1) < epsilon:
            action = np.random.choice(num_actions) # Explorar
        else:
            action = np.argmax(Q_table[state_idx]) # Explotar
            
        next_state_idx, reward, done = env.step(action)
        
        # Actualización de Q-Table mediante Ecuación de Bellman
        mejor_accion_siguiente = np.max(Q_table[next_state_idx])
        Q_table[state_idx, action] = Q_table[state_idx, action] + alpha * (
            reward + gamma * mejor_accion_siguiente - Q_table[state_idx, action]
        )
        
        state_idx = next_state_idx
        total_reward += reward
        
    epsilon = max(0.01, epsilon * decay_rate) # Decaimiento del Epsilon
    recompensas_por_episodio.append(total_reward)

print("Q-Table Entrenada (Filas = Estados, Columnas = Acciones):\n", np.round(Q_table, 2))


In [ ]:
# Visualización del rendimiento por episodio
plt.figure(figsize=(10, 5))
plt.plot(recompensas_por_episodio, color='purple')
plt.title('Recompensa Acumulada por Episodio en Q-Learning')
plt.xlabel('Episodios')
plt.ylabel('Recompensa Total')
plt.grid(True)
plt.show()


## 3. Banco de Preguntas y Respuestas Teórico-Prácticas (Nivel Examen)

### **Pregunta 1 (Beneficio del Semi-Supervisado):**
**Enunciado:** Al evaluar los tres modelos supervisados/semi-supervisados, observamos que entrenar con los 50 representantes seleccionados estratégicamente por K-Means da un mejor resultado que los primeros 50 datos aleatorios (Baseline). Explique conceptualmente a qué se debe esta diferencia.

**Respuesta Correcta y Justificación:**
- **Datos aleatorios (Baseline):** Sufrirá de redundancia y sesgo geográfico. Al ser aleatorios, podemos terminar con muchos ejemplos repetidos del número 1 y ningún ejemplo del número 9 en nuestro conjunto de entrenamiento.
- **Representantes de K-Means:** K-Means distribuye sus 50 centroides de forma que cubran uniformemente todo el espacio de características. Al tomar el punto más cercano a cada centroide, nos aseguramos de elegir un ejemplo prototípico y bien diferenciado de cada uno de los 50 subgrupos, cubriendo todo el espectro de dígitos con el mínimo número de muestras.

---

### **Pregunta 2 (Riesgo del Label Propagation):**
**Enunciado:** Describa el principal riesgo de la **Propagación de Etiquetas** a todo el cluster y proponga una estrategia para mitigar el impacto si el modelo empieza a equivocarse masivamente.

**Respuesta Correcta y Justificación:**
- **Riesgo:** Si el cluster es muy ruidoso o el valor de K es demasiado bajo, elementos que pertenecen a clases distintas terminarán agrupados juntos. Propagar ciegamente la etiqueta del representante a todos los miembros introducirá **etiquetas ruidosas o erróneas** (falsos positivos de etiquetado), degradando el rendimiento del clasificador supervisado final.
- **Mitigación:** En lugar de propagar al $100\%$ de los miembros del cluster, podemos usar la distancia euclidiana y **propagar únicamente al $20\%$ o $30\%$ de los puntos más cercanos al centroide** (el núcleo duro del cluster), dejando el resto como "no etiquetado" o requiriendo una revisión manual adicional.

---

### **Pregunta 3 (Ecuación de Bellman e Hiperparámetros):**
**Enunciado:** Describa el impacto matemático y de comportamiento del agente de navegación si configuramos el factor de descuento **`gamma = 0`** en comparación con **`gamma = 0.95`**.

**Respuesta Correcta y Justificación:**
El factor de descuento $\gamma$ multiplica a la recompensa futura máxima esperada: $\max_{a'} Q(s', a')$.
- **`gamma = 0` (Agente Miope):** La ecuación de Bellman ignorará el término futuro. El agente solo actualizará su Q-table basándose en la recompensa inmediata del siguiente paso ($R$). No aprenderá a planificar a futuro y nunca priorizará dar pasos que den recompensa negativa temporal con tal de alcanzar la meta de $+10$ al final.
- **`gamma = 0.95` (Agente Visionario):** El agente valorará enormemente el camino que lo lleva al objetivo final $+10$, permitiéndole aceptar penalizaciones temporales de paso ($-1$) con tal de encontrar la ruta más corta hacia el Goal.

---

### **Pregunta 4 (Exploración vs. Explotación):**
**Enunciado:** ¿Qué problema ocurriría si dejamos el valor de Epsilon **`epsilon = 0`** constante desde el inicio del entrenamiento del agente de Q-Learning? ¿Y si lo dejamos en **`epsilon = 1.0`**?

**Respuesta Correcta y Justificación:**
- **`epsilon = 0` (Explotación Pura):** El agente nunca tomará acciones aleatorias. Ejecutará siempre la primera acción exitosa que encuentre por casualidad, quedando atrapado en políticas subóptimas y sin descubrir mejores caminos o evitar trampas que no haya visitado inicialmente.
- **`epsilon = 1.0` (Exploración Pura):** El agente tomará acciones 100% aleatorias en cada paso del camino. Aunque llenará la Q-table con valores correctos, se comportará de forma caótica y nunca usará el conocimiento adquirido para completar el laberinto de forma eficiente en los episodios tardíos.

---

### **Pregunta 5 (Pregunta Hipotética sobre Recompensas):**
**Enunciado:** En el Gridworld propuesto, la recompensa por paso normal es de **`-1`**. Si decidimos cambiar esta recompensa de paso por una recompensa de **`0`**, ¿cómo afectaría este cambio al comportamiento de navegación del agente Q-Learning al intentar encontrar el camino óptimo?

**Respuesta Correcta y Justificación:**
Si la recompensa de paso es **0**, el agente no tendrá ningún incentivo para encontrar el camino más corto. Como dar un paso no le cuesta nada, el agente preferirá dar infinitas vueltas en círculos por las celdas seguras sin intentar alcanzar la meta rápidamente. La penalización de paso negativo ($-1$) es crucial para obligar al agente a minimizar los pasos y encontrar la **ruta óptima y más rápida** hacia el Goal.
